In [1]:
import json
import pandas as pd
import numpy as np

# Cargar datos
with open('../data/raw/Datos_SIATA_Aire_pm25.json') as f:
    data = json.load(f)

# Convertir a DataFrame
registros = []
for estacion in data:
    for dato in estacion['datos']:
        registros.append({
            'estacion': estacion['codigoSerial'],
            'nombre': estacion['nombre'],
            'latitud': estacion['latitud'],
            'longitud': estacion['longitud'],
            'fecha': dato['fecha'],
            'pm25': dato['valor'],
            'calidad': dato['calidad']
        })

df = pd.DataFrame(registros)
df['fecha'] = pd.to_datetime(df['fecha'])

print('Datos cargados:', df.shape)

Datos cargados: (183981, 7)


In [2]:
# Limpiar valores inválidos
df_clean = df.copy()

# Reemplazar valores inválidos con NaN
df_clean.loc[df_clean['pm25'] < 0, 'pm25'] = np.nan
df_clean.loc[df_clean['pm25'] > 500, 'pm25'] = np.nan

# Ordenar por estación y fecha
df_clean = df_clean.sort_values(['estacion', 'fecha']).reset_index(drop=True)

# Rellenar NaN con interpolación (usa los valores vecinos para estimar)
df_clean['pm25'] = df_clean.groupby('estacion')['pm25'].transform(
    lambda x: x.interpolate(method='linear', limit=3)
)

# Si quedaron NaN al inicio o final, rellenar con la mediana de la estación
df_clean['pm25'] = df_clean.groupby('estacion')['pm25'].transform(
    lambda x: x.fillna(x.median())
)

print("Valores nulos restantes:", df_clean['pm25'].isna().sum())
print("Shape final:", df_clean.shape)
print("\nEstadísticas PM2.5 limpio:")
print(df_clean['pm25'].describe())

Valores nulos restantes: 0
Shape final: (183981, 7)

Estadísticas PM2.5 limpio:
count    183981.000000
mean         20.457446
std          12.826100
min           0.000000
25%          11.573300
50%          18.000000
75%          26.937400
max         360.000000
Name: pm25, dtype: float64


In [3]:
# Crear features temporales y de series de tiempo
df_features = df_clean.copy()

# Features temporales
df_features['hora'] = df_features['fecha'].dt.hour
df_features['dia_semana'] = df_features['fecha'].dt.dayofweek  # 0=lunes, 6=domingo
df_features['mes'] = df_features['fecha'].dt.month
df_features['es_fin_semana'] = (df_features['dia_semana'] >= 5).astype(int)

# Features de serie de tiempo (valores pasados de la misma estación)
df_features['pm25_lag1'] = df_features.groupby('estacion')['pm25'].shift(1)   # hace 1 hora
df_features['pm25_lag3'] = df_features.groupby('estacion')['pm25'].shift(3)   # hace 3 horas
df_features['pm25_lag6'] = df_features.groupby('estacion')['pm25'].shift(6)   # hace 6 horas
df_features['pm25_lag24'] = df_features.groupby('estacion')['pm25'].shift(24) # hace 24 horas

# Promedio móvil de las últimas 3 y 6 horas
df_features['pm25_media3h'] = df_features.groupby('estacion')['pm25'].transform(
    lambda x: x.shift(1).rolling(3).mean()
)
df_features['pm25_media6h'] = df_features.groupby('estacion')['pm25'].transform(
    lambda x: x.shift(1).rolling(6).mean()
)

# Variable objetivo: PM25 de la próxima hora
df_features['pm25_siguiente'] = df_features.groupby('estacion')['pm25'].shift(-1)

# Eliminar filas con NaN (las primeras y últimas de cada estación)
df_features = df_features.dropna().reset_index(drop=True)

print("Shape con features:", df_features.shape)
print("\nColumnas creadas:")
print(df_features.columns.tolist())

Shape con features: (183456, 18)

Columnas creadas:
['estacion', 'nombre', 'latitud', 'longitud', 'fecha', 'pm25', 'calidad', 'hora', 'dia_semana', 'mes', 'es_fin_semana', 'pm25_lag1', 'pm25_lag3', 'pm25_lag6', 'pm25_lag24', 'pm25_media3h', 'pm25_media6h', 'pm25_siguiente']


In [5]:
# Guardar datos procesados
df_features.to_csv('../data/processed/siata_features.csv', index=False)

print(" Archivo guardado en data/processed/siata_features.csv")
print(f"   {len(df_features):,} registros")
print(f"   {len(df_features.columns)} columnas")
print("\nPrimeras filas:")
df_features.head()

 Archivo guardado en data/processed/siata_features.csv
   183,456 registros
   18 columnas

Primeras filas:


,estacion,nombre,latitud,longitud,fecha,pm25,calidad,hora,dia_semana,mes,es_fin_semana,pm25_lag1,pm25_lag3,pm25_lag6,pm25_lag24,pm25_media3h,pm25_media6h,pm25_siguiente
0,3,Girardota - S.O.S Aburrá Norte,6.379038,-75.450913,2018-08-29 04:00:00,22.0,3.0,4,2,8,0,22.0,24.0,19.0,15.0,24.333333,21.500000,22.0
1,3,Girardota - S.O.S Aburrá Norte,6.379038,-75.450913,2018-08-29 05:00:00,22.0,3.0,5,2,8,0,22.0,27.0,19.0,20.0,23.666667,22.000000,22.0
2,3,Girardota - S.O.S Aburrá Norte,6.379038,-75.450913,2018-08-29 06:00:00,22.0,3.0,6,2,8,0,22.0,22.0,18.0,29.0,22.000000,22.500000,27.0
3,3,Girardota - S.O.S Aburrá Norte,6.379038,-75.450913,2018-08-29 07:00:00,27.0,1.0,7,2,8,0,22.0,22.0,24.0,22.0,22.000000,23.166667,38.0
4,3,Girardota - S.O.S Aburrá Norte,6.379038,-75.450913,2018-08-29 08:00:00,38.0,1.0,8,2,8,0,27.0,22.0,27.0,29.0,23.666667,23.666667,32.0
